# Tutorial 2: from an imaginary phonon to its isotropy subgroups

**For:** people working on structural phase transitions and structure searches, who have an unstable phonon and want the complete list of lower-symmetry phases it can lead to.
**Time:** about 15 minutes.
**Requirements:** CrystOD >= 0.4.0 (`pip install CrystOD`), which installs phonopy, spglib and pandas. No PySCF is needed.

What this notebook does:

1. enumerates the isotropy subgroups of one irrep with `crystod.group.isotropy_subgroups`, without any structure file,
2. does the same for the imaginary R-point phonon of SrTiO3 straight from the phonopy object (`crystod.phonon.imaginary_mode_subgroups`),
3. scans every q point the supercell resolves with `scan_imaginary_modes`,
4. shows why freezing in one eigenvector at a time misses R-3c and Imma,
5. generates the distorted candidate structures with `crystod-phonon --subgroup --modulate` and re-measures their space groups with spglib.

In [1]:
import crystod

crystod.__version__   # 0.4.0 or later is required

'0.4.0'

In [2]:
import os
import tempfile

workdir = tempfile.mkdtemp(prefix="crystod_tutorial_")
os.chdir(workdir)   # everything this notebook writes lands here, not next to the notebook
workdir

'/var/folders/z9/fy920zrj7jqf_3c45v8_s9400000gn/T/crystod_tutorial_igbiboj3'

## Isotropy subgroups of one irrep

In Landau theory a continuous phase transition is driven by an order parameter that transforms as one irrep of the parent space group. The irrep fixes *which* distortions are allowed; the *direction* of the order parameter inside the irrep space fixes the symmetry that survives, the isotropy subgroup. A three-dimensional irrep therefore has several isotropy subgroups, one per inequivalent direction.

`R4+` of Pm-3m is the antiphase rotation of the octahedra of a perovskite (Glazer's minus tilts, with the B cation at the origin). `isotropy_subgroups` lists its directions, the space group each condenses into, the multiplication of the primitive cell (`size`) and the index of the subgroup in the parent; the order and the labels are those of ISOTROPY's ISOSUBGROUP.

In [3]:
import pandas as pd
from crystod.group import isotropy_subgroups

subgroups = isotropy_subgroups("Pm-3m", "R4+")
pd.DataFrame(
    [
        {"direction": s.label, "subgroup": f"{s.symbol} (No. {s.number})", "cell size": s.size,
         "index": s.index, "free parameters": s.n_free}
        for s in subgroups
    ]
)

,direction,subgroup,cell size,index,free parameters
0,"R4+(0,0,a)",I4/mcm (No. 140),2,6,1
1,"R4+(a,a,a)",R-3c (No. 167),2,8,1
2,"R4+(0,a,a)",Imma (No. 74),2,12,1
3,"R4+(0,a,b)",C2/m (No. 12),2,24,2
4,"R4+(a,a,b)",C2/c (No. 15),2,24,2
5,"R4+(a,b,c)",P-1 (No. 2),2,48,3


Reading the table as tilt systems: `(0,0,a)` is a rotation about one cubic axis (a0a0c-, I4/mcm, the low-temperature phase of SrTiO3), `(a,a,a)` the equal rotation about all three (a-a-a-, R-3c, as in LaAlO3), `(0,a,a)` about two (a0b-b-, Imma); the remaining directions have two or three independent amplitudes and end in monoclinic or triclinic groups.

A single direction can also be resolved by name, and then the entry carries the conventional cell of the subgroup in the parent setting (`basis` rows in units of the cubic cell, and `origin`), which is what a VASP or CIF file of the daughter phase needs.

In [4]:
tetragonal = isotropy_subgroups("Pm-3m", "R4+", order_parameter=["0", "0", "a"])[0]
print(tetragonal)
print("conventional basis of I4/mcm in the cubic setting (rows):")
print(tetragonal.basis)
print("origin:", tetragonal.origin)

R4+(0,0,a) -> I4/mcm (No. 140), size 2, index 6
conventional basis of I4/mcm in the cubic setting (rows):
[[-1.  0.  1.]
 [ 1.  0.  1.]
 [ 0.  2.  0.]]
origin: [ 0.  0. -0.]


## The imaginary mode of SrTiO3

Now the same question is asked of a real phonon calculation. The phonopy object is the one of Tutorial 1 (bundled 4x4x4 force sets of cubic SrTiO3; `primitive_matrix="auto"` keeps the instability at its true q point). `imaginary_mode_subgroups` labels every level at q below the `threshold` (-0.1 THz by default) and enumerates the isotropy subgroups of its irrep in one call.

One remark on the label: in the bundled POSCAR the Sr atom sits at the origin, and in that setting the octahedral rotation transforms as `R5-`; with Ti at the origin the same mode is `R4+`. The two labels describe the same distortion and have the same subgroup list.

In [5]:
import phonopy
from crystod.examples import example_path

ph = phonopy.load(
    supercell_matrix=[4, 4, 4],
    primitive_matrix="auto",
    unitcell_filename=example_path("221_PPOSCAR_SrTiO3"),
    force_sets_filename=example_path("FORCE_SETS_SrTiO3"),
)
print(ph.primitive_symmetry.dataset.international, "-", len(ph.primitive), "atoms in the primitive cell,",
      3 * len(ph.primitive), "phonon branches")

Pm-3m - 5 atoms in the primitive cell, 15 phonon branches


In [6]:
from crystod.phonon import imaginary_mode_subgroups

r_results = imaginary_mode_subgroups(ph, [0.5, 0.5, 0.5])
for result in r_results:
    mode = result.mode
    print(f"{mode.frequency:.4f} THz  {'+'.join(mode.labels)}  (degeneracy {mode.degeneracy}) "
          f"in {result.space_group} (No. {result.space_group_number})")
    for sub in result.subgroups:
        print(f"    {sub.label:<12} -> {sub.symbol:<7} No. {sub.number:<4} size {sub.size}  index {sub.index}")

-1.0867 THz  R5-  (degeneracy 3) in Pm-3m (No. 221)
    R5-(0,0,a)   -> I4/mcm  No. 140  size 2  index 6
    R5-(a,a,a)   -> R-3c    No. 167  size 2  index 8
    R5-(0,a,a)   -> Imma    No. 74   size 2  index 12
    R5-(0,a,b)   -> C2/m    No. 12   size 2  index 24
    R5-(a,a,b)   -> C2/c    No. 15   size 2  index 24
    R5-(a,b,c)   -> P-1     No. 2    size 2  index 48


## Scan every q point the supercell resolves

A supercell calculation resolves exactly the q points that fold onto Gamma of the supercell, 64 of them for a 4x4x4 cell; `commensurate_qpoints` lists them. `scan_imaginary_modes` runs the analysis at every one of them, analyzes each star only once, sorts the levels most unstable first, and never dies part-way: a q point that cannot be labeled is skipped with a warning, and a level whose irrep has no tabulated subgroups (a symmetry line or plane) comes back with an empty `subgroups` and the reason in `errors`.

In [7]:
from crystod.phonon import commensurate_qpoints, scan_imaginary_modes

qpoints = commensurate_qpoints(ph)
print(len(qpoints), "commensurate q points, the first four:", qpoints[:4])

results = scan_imaginary_modes(ph)
print(len(results), "imaginary level(s) below -0.1 THz")
for result in results:
    print(result.mode, "at", result.mode.qpoint_label or result.mode.qpoint)
    print("   ", ", ".join(f"{s.label} -> {s.symbol}" for s in result.subgroups) or result.errors)

64 commensurate q points, the first four: [(0.0, 0.0, 0.0), (0.0, 0.0, 0.25), (0.0, 0.25, 0.0), (0.25, 0.0, 0.0)]


1 imaginary level(s) below -0.1 THz
modes 1,2,3: -1.0867 THz  R5- at R
    R5-(0,0,a) -> I4/mcm, R5-(a,a,a) -> R-3c, R5-(0,a,a) -> Imma, R5-(0,a,b) -> C2/m, R5-(a,a,b) -> C2/c, R5-(a,b,c) -> P-1


For SrTiO3 the R-point triplet is the only instability, so the scan and the single-q call agree; for a structure with several unstable q points the scan is the one that finds them all.

## Why one frozen eigenvector is not enough

The common shortcut is to freeze in the imaginary eigenvectors one at a time and relax each result. For a degenerate level that explores one direction per eigenvector, and *which* direction depends on the arbitrary orthonormal basis the diagonalizer happens to return for the triplet. The experiment below uses phonopy's own `run_modulations` to freeze in each of the three `R5-` eigenvectors alone (in the 2x2x2 supercell the R point needs) and lets spglib measure the space group of the result.

In [8]:
import spglib


def space_group_of(cell, symprec=1e-3):
    return spglib.get_spacegroup((cell.cell, cell.scaled_positions, cell.numbers), symprec=symprec)


soft = r_results[0].mode      # the R5- triplet: bands 1, 2, 3
for band in soft.band_indices:
    # phonopy counts bands from 0; amplitude 0.3 Angstrom, phase 0
    ph.run_modulations(dimension=[2, 2, 2], phonon_modes=[[list(soft.qpoint), band - 1, 0.3, 0.0]])
    distorted = ph.modulation.modulated_supercells[0]
    print(f"eigenvector {band} frozen in alone  ->  {space_group_of(distorted)}")

eigenvector 1 frozen in alone  ->  I4/mcm (140)
eigenvector 2 frozen in alone  ->  C2/m (12)
eigenvector 3 frozen in alone  ->  C2/m (12)


One eigenvector gives I4/mcm and the other two give C2/m; R-3c and Imma, the tilt systems a perovskite structure search has to try before it can claim a ground state, never appear. They correspond to the directions `(a,a,a)` and `(0,a,a)` in the symmetry-adapted basis of the irrep, and the eigenvectors of a numerical diagonalization are not that basis. The enumeration of `isotropy_subgroups` does not depend on any basis, which is why it is complete.

## Generate the candidate structures

`crystod-phonon --subgroup` prints the enumeration above from the command line, and `--modulate` adds the `--modulation` step for every direction: the combination of the degenerate modes that realizes each direction is not assumed but *measured*, by generating candidate combinations and matching the spglib space group, cell size and index of each against the table. The bundled inputs are copied into the working directory with `crystod.examples.copy_example_files` (which is what `--example SrTiO3_subgroup` does), and the command runs there.

In [9]:
import subprocess
import sys

from crystod.examples import copy_example_files, get_example

copy_example_files(get_example("crystod-phonon", "SrTiO3_subgroup"), workdir)   # 221_PPOSCAR_SrTiO3, FORCE_SETS
command = [sys.executable, "-m", "crystod.cli.phonon",                          # == crystod-phonon
           "--subgroup", "--dim", "4 4 4", "-c", "221_PPOSCAR_SrTiO3", "--qpoint", "R", "--modulate"]
run = subprocess.run(command, cwd=workdir, capture_output=True, text=True, check=True)
print(run.stdout)


* Parent structure *
Pm-3m (No. 221)

* Imaginary mode at q = (0.5, 0.5, 0.5) (R) *
mode 1, 2, 3: -1.086709 THz, irrep R5- (degeneracy 3)

irrep                subgroup           size  index
R5-(0,0,a)           140 I4/mcm         2     6    
R5-(a,a,a)           167 R-3c           2     8    
R5-(0,a,a)           74 Imma            2     12   
R5-(0,a,b)           12 C2/m            2     24   
R5-(a,a,b)           15 C2/c            2     24   
R5-(a,b,c)           2 P-1              2     48   

* Distorted structures (amplitude 0.3 A) *
R5-(0,0,a)             I4/mcm     -> MPOSCAR_R_R5-_0-0-a_I4mcm
    crystod-phonon --modulation -c 221_PPOSCAR_SrTiO3 --dim "4 4 4" --qpoint 0.5 0.5 0.5 --mode 1 --amplitude 0.3
R5-(0,a,a)             Imma       -> MPOSCAR_R_R5-_0-a-a_Imma
    crystod-phonon --modulation -c 221_PPOSCAR_SrTiO3 --dim "4 4 4" --qpoint 0.5 0.5 0.5 --mode 1 2 --amplitude 0.3 0.3
R5-(0,a,b)             C2/m       -> MPOSCAR_R_R5-_0-a-b_C2m
    crystod-phonon --modulation 

One POSCAR per direction, named `MPOSCAR_{q}_{irrep}_{direction}_{spacegroup}`, each with the `--modulation` command that reproduces it. Reading them back and asking spglib for their space groups confirms that all six directions were realized, including the two the single-eigenvector experiment could not reach. These files are the starting points of the relaxations of a structure search.

In [10]:
from phonopy.interface.vasp import read_vasp

rows = []
for name in sorted(f for f in os.listdir(workdir) if f.startswith("MPOSCAR_")):
    cell = read_vasp(name)
    rows.append({"file": name, "atoms": len(cell.numbers), "space group (spglib)": space_group_of(cell)})
pd.DataFrame(rows)

,file,atoms,space group (spglib)
0,MPOSCAR_R_R5-_0-0-a_I4mcm,40,I4/mcm (140)
1,MPOSCAR_R_R5-_0-a-a_Imma,40,Imma (74)
2,MPOSCAR_R_R5-_0-a-b_C2m,40,C2/m (12)
3,MPOSCAR_R_R5-_a-a-a_R-3c,40,R-3c (167)
4,MPOSCAR_R_R5-_a-a-b_C2c,40,C2/c (15)
5,MPOSCAR_R_R5-_a-b-c_P-1,40,P-1 (2)


## Where next

- [27. Subgroups from imaginary modes](https://mochizuki-tus.github.io/CrystOD/crystod-phonon.html#subgroups-from-imaginary-modes-subgroup) and [25. Phonon modulation](https://mochizuki-tus.github.io/CrystOD/crystod-phonon.html#phonon-modulation-modulation) in the manual: multi-arm stars (the M point of a perovskite), amplitudes, and the `--modulation` command by hand.
- [13. Isotropy subgroups](https://mochizuki-tus.github.io/CrystOD/crystod-group.html#isotropy-subgroups-parent): `crystod-group --parent Pm-3m --irrep R4+` is the command-line form of the first table, and `--irrep X3- X2+` enumerates coupled order parameters.
- [Isotropy subgroups (theory)](https://mochizuki-tus.github.io/CrystOD/theory-isotropy-subgroups.html): how the directions are enumerated and validated against ISOSUBGROUP.
- `example/35_python_api/macer_subgroup_bridge.py` in the repository: this workflow as a script for a structure search.
- After relaxing the daughters, [16. Symmetry-mode analysis](https://mochizuki-tus.github.io/CrystOD/crystod-group.html#symmetry-mode-analysis-supergroup-cif) decomposes the relaxed structure into irrep amplitudes, and Tutorial 1 labels its phonons.